# EDA de transferencias de autos 2026 - Proyecto Medallion

- **Tema:** carga a Bronze + EDA (Exploratory Data Analysis) con SQL
- **Dataset:** transferencias de automotores del DNRPA (Registro Nacional de la Propiedad del Automotor), 2026, un CSV por mes
- **Tabla Bronze que vamos a crear:** `bootcamp.bronze.transferencias_autos_bronze`

**Instrucciones**
1. Usar `%sql` al inicio de cada celda SQL
2. Documentar los hallazgos en comentarios (`-- Hallazgos:`)
3. Subir las soluciones a GitHub

> **Importante:** un buen EDA es la base de un buen pipeline. Los hallazgos de acá definen las reglas de Silver.
> **Antes de empezar:** confirmá la ruta de los CSV con la celda 0.1. Si están en otra carpeta, cambiá `/Volumes/bootcamp/landing/proyecto/` en las celdas de la Parte 0.

## Contexto del dataset
- **Fuente:** Ministerio de Justicia, DNRPA. Dataset "Transferencias de autos" (portal de datos abiertos). Un ZIP por año; adentro hay un CSV por mes (`dnrpa-transferencias-autos-AAAAMM.csv`).
- **Qué es una fila:** una transferencia (cambio de titular) de un automotor ya inscripto, **por titular**. Si el auto pasa a dos dueños es probable que aparezcan dos filas (columna `titular_porcentaje_titularidad`). A verificar en la Parte 5.
- **Qué NO tiene:** precio y patente. No se puede seguir un mismo auto en el tiempo ni analizar valores, solo volumen y perfil.
- **Pregunta de negocio guía:** ¿qué autos (marca y modelo) se transfieren más en cada mes, y qué tan estable es ese ranking para evaluar si la tendencia seguiría? Útil, por ejemplo, para concesionarias.
- **Regla de Bronze:** todo como texto (`inferColumnTypes => false`), sin castear y sin filtrar. El tipado y la limpieza van en Silver.

---
# PARTE 0: Carga de datos y prueba de lectura
**Objetivo:** antes de crear la tabla Bronze, probar cómo vienen los CSV (todos) para detectar problemas de lectura.
Orden: ver los archivos, leer uno, leer todos con controles, y recién ahí crear la tabla.

### Ejercicio 0.1: Ver qué archivos hay en el Volume

¿Están los 9 CSV (de `202601` a `202609`)? ¿Hay otros archivos mezclados en la carpeta (por ejemplo Parquet de otro proyecto)?

*Pista: el comando `LIST` sobre la ruta del Volume. Fijate en el nombre y el tamaño de cada archivo.*

In [0]:
%sql
LIST '/Volumes/bootcamp/landing/proyecto/';

-- Hallazgos:
-- 

### Ejercicio 0.2: Leer UN archivo dejando que Spark infiera los tipos

Leé el CSV de enero con `read_files` y mirá cómo infiere los tipos de las columnas.

*Pista: `read_files('ruta', format => 'csv', header => true)`. Después `DESCRIBE QUERY` para ver el esquema inferido. Prestá atención a columnas como `automotor_tipo_codigo`, `registro_seccional_codigo` o `titular_domicilio_provincia_id`: ¿qué pasa con los ceros a la izquierda (`05`, `02`)?*

In [0]:
%sql
SELECT *
FROM read_files(
  '/Volumes/bootcamp/landing/proyecto/dnrpa-transferencias-autos-202602.csv',
  format => 'csv',
  header => true
)
LIMIT 10;

-- Hallazgos:
-- Hay algunas claves que vienen por defecto: registro_seccional_codigo → podría convertirse en registro_seccional_id
--automotor_tipo_codigo → seria automotor_tipo_id y su automotor_tipo (categorizaciones) sería el campo automotor_tipo_descripcion.

--Lo mismo pasa con: automotor_marca_codigo (mezcla de numeros y letras) y respectivo campo: automotor_marca_descripcion.

--Automotor_modelo_codigo (respectivo automotor_modelo_descripcion)


In [0]:
%sql
DESCRIBE QUERY
SELECT *
FROM read_files(
  '/Volumes/bootcamp/landing/proyecto/dnrpa-transferencias-autos-202602.csv', --cambiar acá la terminacion del csv y probar
  format => 'csv',
  header => true
);

-- Hallazgos (¿qué columnas infirió como número? ¿se pierden ceros a la izquierda?):
--Columnas inferidas como numero: 

--tramite_fecha	date
--titular_porcentaje_titularidad	double
--registro_seccional_codigo	int
--automotor_anio_modelo	int
--automotor_uso_codigo	int
--titular_domicilio_provincia_id	int
--titular_pais_nacimiento_id	int

-- 

### Ejercicio 0.3: Leer el mismo archivo todo como texto

En Bronze queremos **todo tal cual llega**. Repetí la lectura desactivando la inferencia de tipos y comparalo con el ejercicio anterior.

*Pista: agregá `inferColumnTypes => false`. Todas las columnas deberían quedar como `string`.*

In [0]:
%sql
DESCRIBE QUERY
SELECT *
FROM read_files(
  '/Volumes/bootcamp/landing/proyecto/dnrpa-transferencias-autos-202601.csv',
  format => 'csv',
  header => true,
  inferColumnTypes => false --aclarar que no infiera los datos
);

-- Hallazgos:
-- 26 columnas, todas detectadas como string

### Ejercicio 0.4: Probar TODOS los CSV a la vez, uno por uno

¿Cuántas filas tiene cada archivo? ¿El rango de `tramite_fecha` de cada archivo corresponde a su mes? ¿Hay filas que no se pudieron parsear?

*Pista: leé la carpeta completa con `pathGlobFilter` para quedarte solo con estos CSV. Usá `_metadata.file_name` para agrupar por archivo y `_rescued_data` (columna que Spark completa cuando una fila no encaja con el esquema).*

> Ojo con el archivo de **febrero**: tiene fecha de modificación y tamaño distintos a los demás. Fijate si el conteo y las fechas son coherentes.

In [0]:
%sql
SELECT
  _metadata.file_name                       AS archivo,
  COUNT(*)                                  AS filas,
  MIN(tramite_fecha)                        AS fecha_min,
  MAX(tramite_fecha)                        AS fecha_max,
  COUNT_IF(_rescued_data IS NOT NULL)       AS filas_no_parseadas
FROM read_files(
  '/Volumes/bootcamp/landing/proyecto/',
  format => 'csv',
  header => true,
  inferColumnTypes => false,
  pathGlobFilter => 'dnrpa-transferencias-autos-*.csv'
)
GROUP BY _metadata.file_name
ORDER BY archivo;

-- Hallazgos:
-- 

**Prueba con Python para verificar los headers**

In [0]:
%python
ruta = "/Volumes/bootcamp/landing/proyecto/"
archivos = sorted([f.path for f in dbutils.fs.ls(ruta)
                   if f.name.startswith("dnrpa-transferencias-autos-") and f.name.endswith(".csv")])

# columnas de cada archivo, tal cual vienen en el encabezado
cols = {}
for a in archivos:
    cols[a.split("/")[-1]] = spark.read.option("header", True).csv(a).columns

ref_nombre = list(cols)[0]
ref = cols[ref_nombre]
print(f"Referencia: {ref_nombre} -> {len(ref)} columnas\n")

for nombre, c in cols.items():
    faltan  = [x for x in ref if x not in c]
    sobran  = [x for x in c if x not in ref]
    orden   = (c != ref) and not faltan and not sobran
    estado  = "OK" if c == ref else "DIFERENTE"
    print(f"{nombre}: {len(c)} columnas -> {estado}")
    if faltan: print("   faltan:", faltan)
    if sobran: print("   sobran:", sobran)
    if orden:  print("   mismas columnas pero en otro orden")

### Ejercicio 0.5: ¿El contenido de cada archivo es del mes que dice su nombre?

Comparé el período del nombre del archivo (`AAAAMM`) con el mes de `tramite_fecha`. Si un archivo trae meses mezclados o de otro período, hay que saberlo antes de cargar.

*Pista: `regexp_extract(_metadata.file_name, '([0-9]{6})', 1)` para sacar el período y `substr(tramite_fecha, 1, 7)` para el mes del dato.*

In [0]:
%sql
SELECT
  regexp_extract(_metadata.file_name, '([0-9]{6})', 1) AS periodo_archivo,
  substr(tramite_fecha, 1, 7)                          AS mes_en_los_datos,
  COUNT(*)                                             AS filas
FROM read_files(
  '/Volumes/bootcamp/landing/proyecto/',
  format => 'csv',
  header => true,
  inferColumnTypes => false,
  pathGlobFilter => 'dnrpa-transferencias-autos-*.csv'
)
GROUP BY 1, 2
ORDER BY 1, 2;

-- Hallazgos:
-- 

### Ejercicio 0.6: Controles de sanidad de la lectura

Tres controles rápidos por archivo:
1. **Encoding:** ¿se ven caracteres rotos (`Ã`, `�`) en textos con tildes (por ejemplo `Nº`, `Física`)?
2. **Columnas corridas:** si una coma dentro de un texto desplazó las columnas, `automotor_origen` dejaría de tener solo `Nacional`, `Importado` y `Protocolo 21`.
3. **Tipo de persona:** debería ser solo `Física` o `Jurídica`.

*Pista: `COUNT_IF(...)` con `LIKE` y `NOT IN`. Acordate de que `NOT IN` no cuenta los NULL: agregá `IS NULL` explícito.*

In [0]:
%sql
SELECT
  _metadata.file_name AS archivo,
  COUNT_IF(registro_seccional_descripcion LIKE '%Ã%' OR registro_seccional_descripcion LIKE '%�%'
           OR titular_tipo_persona LIKE '%Ã%' OR titular_tipo_persona LIKE '%�%')          AS posible_mal_encoding,
  COUNT_IF(automotor_origen IS NULL
           OR automotor_origen NOT IN ('Nacional', 'Importado', 'Protocolo 21'))            AS origen_inesperado,
  COUNT_IF(titular_tipo_persona IS NULL
           OR titular_tipo_persona NOT LIKE 'F%' AND titular_tipo_persona NOT LIKE 'J%')   AS persona_inesperada
FROM read_files(
  '/Volumes/bootcamp/landing/proyecto/',
  format => 'csv',
  header => true,
  inferColumnTypes => false,
  pathGlobFilter => 'dnrpa-transferencias-autos-*.csv'
)
GROUP BY _metadata.file_name
ORDER BY archivo;

-- Hallazgos (si algún valor no es 0, investigar ese archivo antes de seguir):
-- 

### Ejercicio 0.7: Crear la tabla Bronze

Con la lectura probada, creá **una sola tabla** `bootcamp.bronze.transferencias_autos_bronze` con todos los CSV.

**¿Una tabla con todo o una tabla por mes?** Recomendación: **una sola tabla**.
- Los 9 archivos tienen el mismo esquema. Las tablas por mes serían copias de la misma estructura.
- La columna `_archivo_origen` conserva de qué archivo vino cada fila: no se pierde la trazabilidad por tener todo junto.
- La pregunta de negocio (ranking y tendencia por mes) necesita los meses **juntos**. Con tablas separadas habría que hacer `UNION ALL` de N tablas en cada consulta, y N crece todos los meses (y todos los años).
- Un mes nuevo es una carga adicional sobre la misma tabla, no una tabla nueva.
- Tablas separadas tendrían sentido si los archivos tuvieran esquemas distintos, y acá no es el caso.

**Decisiones de Bronze en esta carga**
- Todo texto (`inferColumnTypes => false`): así no se pierden ceros a la izquierda ni se rompe por un valor raro.
- **Sin filtros** (a diferencia del ejemplo de propiedades, donde se filtraba `url LIKE 'https%'`): Bronze guarda todo tal cual; los filtros van en Silver.
- Columnas de auditoría: `_archivo_origen` e `_ingesta_ts`. `_rescued_data` queda por si hubo filas con problemas.

*Pista: `CREATE OR REPLACE TABLE ... AS SELECT *, _metadata.file_path AS _archivo_origen, current_timestamp() AS _ingesta_ts FROM read_files(...)`. Es seguro correrlo de nuevo: reemplaza la tabla completa.*

In [0]:
%sql
CREATE OR REPLACE TABLE bootcamp.bronze.transferencias_autos_bronze
COMMENT 'Bronze: transferencias de automotores (DNRPA) 2026 Hasta Septiembre. Todo texto, sin filtros. Una fila por titular.'
AS
SELECT
  *,
  _metadata.file_path  AS _archivo_origen,
  current_timestamp()  AS _ingesta_ts
FROM read_files(
  '/Volumes/bootcamp/landing/proyecto/',
  format => 'csv',
  header => true,
  inferColumnTypes => false,
  pathGlobFilter => 'dnrpa-transferencias-autos-*.csv'
);

-- Hallazgos:
-- hehco antes (ya asentado, no vuelvo a correr)

### Ejercicio 0.8: Verificar la tabla Bronze

¿La cantidad de filas de la tabla coincide con la suma de filas de los archivos que viste en el ejercicio 0.4? ¿Hay filas por archivo?

*Pista: `COUNT(*)` por `_archivo_origen` y el total con `SUM(COUNT(*)) OVER()`.*

In [0]:
%sql
SELECT
  _archivo_origen,
  COUNT(*)                              AS filas,
  SUM(COUNT(*)) OVER ()                 AS total_tabla
FROM bootcamp.bronze.transferencias_autos_bronze --ok
GROUP BY _archivo_origen
ORDER BY _archivo_origen;

-- Hallazgos (el total debe coincidir con la suma del ejercicio 0.4):
-- 

> **Agregar meses nuevos más adelante** (tarea futura, no se ejecuta ahora): para sumar, por ejemplo, octubre, se carga ese CSV como una carga adicional sobre la misma tabla (`INSERT INTO ... SELECT ... FROM read_files('.../dnrpa-transferencias-autos-202610.csv', ...)` o `COPY INTO`, que registra qué archivos ya cargó y evita duplicados). Si se vuelve a correr el `INSERT` con el mismo archivo se duplican las filas: por eso conviene `COPY INTO`.

---
# PARTE 1: Exploración inicial
Dataset: `bootcamp.bronze.transferencias_autos_bronze`

**Importante en bronze: dejar TODO tal cual te llega**

### Ejercicio 1.1: Verificar datos disponibles

¿Cuántos registros hay en total en la tabla Bronze?

*Pista: usá `COUNT(*)`*

In [0]:
%sql
SELECT COUNT(*) AS filas
FROM bootcamp.bronze.transferencias_autos_bronze;

-- Hallazgos:
-- 1378157 filas de todo el 2026

### Ejercicio 1.2: Explorar estructura

¿Qué columnas tiene la tabla? ¿Qué tipos de datos tienen? ¿Cuáles son las columnas de auditoría?

*Pista: usá `DESCRIBE TABLE`*

In [0]:
%sql
DESCRIBE TABLE bootcamp.bronze.transferencias_autos_bronze;

-- Hallazgos:
-- 

### Ejercicio 1.3: Ver muestra de datos

Mostrá 10 filas con las columnas más importantes: `tramite_tipo, tramite_fecha, fecha_inscripcion_inicial, registro_seccional_provincia, automotor_origen, automotor_anio_modelo, automotor_tipo_descripcion, automotor_marca_descripcion, automotor_modelo_descripcion, titular_tipo_persona, titular_genero, titular_anio_nacimiento, titular_porcentaje_titularidad`.

*Pista: `SELECT` con columnas específicas y `LIMIT 10`*

In [0]:
%sql
SELECT
  tramite_tipo,
  tramite_fecha,
  fecha_inscripcion_inicial,
  registro_seccional_provincia,
  automotor_origen,
  automotor_anio_modelo,
  automotor_tipo_descripcion,
  automotor_marca_descripcion,
  automotor_modelo_descripcion,
  titular_tipo_persona,
  titular_genero,
  titular_anio_nacimiento,
  titular_porcentaje_titularidad
FROM bootcamp.bronze.transferencias_autos_bronze
LIMIT 10;

-- Hallazgos:
-- 

### Ejercicio 1.3.1: ¿Que variables tengo en columnas categoricas?



In [0]:
%sql
SELECT columna, 
COUNT(DISTINCT valor) AS distintos
FROM (
  SELECT tramite_tipo, tramite_fecha, fecha_inscripcion_inicial, registro_seccional_codigo,
         registro_seccional_descripcion, registro_seccional_provincia, automotor_origen,
         automotor_anio_modelo, automotor_tipo_codigo, automotor_tipo_descripcion,
         automotor_marca_codigo, automotor_marca_descripcion, automotor_modelo_codigo,
         automotor_modelo_descripcion, automotor_uso_codigo, automotor_uso_descripcion,
         titular_tipo_persona, titular_domicilio_localidad, titular_domicilio_provincia,
         titular_genero, titular_anio_nacimiento, titular_pais_nacimiento,
         titular_porcentaje_titularidad, titular_domicilio_provincia_id, titular_pais_nacimiento_id
  FROM bootcamp.bronze.transferencias_autos_bronze
)
UNPIVOT INCLUDE NULLS (valor FOR columna IN (
  tramite_tipo, tramite_fecha, fecha_inscripcion_inicial, registro_seccional_codigo,
  registro_seccional_descripcion, registro_seccional_provincia, automotor_origen,
  automotor_anio_modelo, automotor_tipo_codigo, automotor_tipo_descripcion,
  automotor_marca_codigo, automotor_marca_descripcion, automotor_modelo_codigo,
  automotor_modelo_descripcion, automotor_uso_codigo, automotor_uso_descripcion,
  titular_tipo_persona, titular_domicilio_localidad, titular_domicilio_provincia,
  titular_genero, titular_anio_nacimiento, titular_pais_nacimiento,
  titular_porcentaje_titularidad, titular_domicilio_provincia_id, titular_pais_nacimiento_id))
GROUP BY columna
ORDER BY distintos;

**Verificar que catgorias hay en los campos con string dese la lectura de csv**

In [0]:

%sql
WITH valores AS (
  SELECT columna, valor, COUNT(*) AS filas
  FROM bootcamp.bronze.transferencias_autos_bronze
  UNPIVOT INCLUDE NULLS (valor FOR columna IN (
    tramite_tipo, fecha_inscripcion_inicial, registro_seccional_descripcion,
    registro_seccional_provincia, automotor_origen, automotor_tipo_codigo,
    automotor_tipo_descripcion, automotor_marca_codigo, automotor_marca_descripcion,
    automotor_modelo_codigo, automotor_modelo_descripcion, automotor_uso_descripcion,
    titular_tipo_persona, titular_domicilio_localidad, titular_domicilio_provincia,
    titular_genero, titular_anio_nacimiento, titular_pais_nacimiento))
  GROUP BY columna, valor
),
con_cardinalidad AS (
  SELECT *, COUNT(*) OVER (PARTITION BY columna) AS distintos
  FROM valores
)
SELECT columna, distintos, valor, filas
FROM con_cardinalidad
WHERE distintos <= 10000          -- cambiá este número si querés ver más o menos columnas
ORDER BY distintos, columna, filas DESC;

--TENER CONOCIMIENTO DE QUE COLUMNAS SON CATEGORICAS ACÁ: 
-- tramite_tipo
-- fecha_inscripcion_inicial
-- registro_seccional_descripcion
-- registro_seccional_provincia
-- automotor_origen
-- automotor_tipo_codigo
-- automotor_tipo_descripcion
-- automotor_marca_codigo
-- automotor_marca_descripcion
-- automotor_modelo_codigo
-- automotor_modelo_descripcion
-- automotor_uso_descripcion
-- titular_tipo_persona
-- titular_domicilio_localidad
-- titular_domicilio_provincia
-- titular_genero
-- titular_anio_nacimiento
-- titular_pais_nacimiento

SELECT DISTINCT tramite_tipo  FROM 
bootcamp.bronze.transferencias_autos_bronze; --25 valores distintos

---------------------------------OBSERVACIÓN FECHA_INSCRIPCION_INICIAL ↓↓--------------
SELECT count(DISTINCT fecha_inscripcion_inicial) 
FROM 
bootcamp.bronze.transferencias_autos_bronze; --15595 fechas 
---
SELECT --averiguar si hay formato raro en fechas: porque lo tomaba originalmente como string
  regexp_replace(fecha_inscripcion_inicial, '[0-9]', '9') AS formato,
  COUNT(*)                                                AS filas,
  MIN(fecha_inscripcion_inicial)                          AS ejemplo_min,
  MAX(fecha_inscripcion_inicial)                          AS ejemplo_max
FROM bootcamp.bronze.transferencias_autos_bronze
GROUP BY 1
ORDER BY filas DESC;

--RESULTADO: 8 FILAS NULAS: 
--Doble comprobación: 

SELECT _archivo_origen, tramite_tipo, tramite_fecha, automotor_marca_descripcion,
       automotor_modelo_descripcion, automotor_anio_modelo, titular_tipo_persona
FROM bootcamp.bronze.transferencias_autos_bronze
WHERE TRIM(fecha_inscripcion_inicial) = '';

SELECT * from bootcamp.bronze.transferencias_autos_bronze 
where TRIM(fecha_inscripcion_inicial) = '';  --esta condición y no IS NULL, porque no las encuentra con esa cond.
---------------------------------↑↑OBSERVACIÓN FECHA_INSCRIPCION_INICIAL ↑↑--------------

SELECT DISTINCT registro_seccional_descripcion
FROM 
bootcamp.bronze.transferencias_autos_bronze; --852 distintos

SELECT DISTINCT registro_seccional_provincia
FROM 
bootcamp.bronze.transferencias_autos_bronze;  --25 correspondientes a provincia + CABA + NULL

SELECT DISTINCT automotor_origen
FROM 
bootcamp.bronze.transferencias_autos_bronze; --3: Protocolo 21, Importado, Nacional


SELECT DISTINCT automotor_tipo_codigo
FROM bootcamp.bronze.transferencias_autos_bronze; --119

SELECT DISTINCT automotor_tipo_descripcion
FROM bootcamp.bronze.transferencias_autos_bronze; -4681 (considerar CASE para agrupar)

SELECT DISTINCT automotor_marca_codigo
FROM bootcamp.bronze.transferencias_autos_bronze;  --707 

SELECT DISTINCT automotor_marca_descripcion
FROM bootcamp.bronze.transferencias_autos_bronze;  --campo marca: CONSIDERAR CASE (1454 distintos)

SELECT DISTINCT automotor_modelo_codigo
FROM bootcamp.bronze.transferencias_autos_bronze; --2368 

SELECT COUNT(DISTINCT automotor_modelo_descripcion)
FROM bootcamp.bronze.transferencias_autos_bronze; --44608

SELECT DISTINCT automotor_uso_codigo
FROM bootcamp.bronze.transferencias_autos_bronze; --4 VALORES, 1 NULO

----------------uso_codigo mapeo------------------------

SELECT DISTINCT automotor_uso_codigo, automotor_uso_descripcion
FROM bootcamp.bronze.transferencias_autos_bronze; --aprece 1 como 'Privado y '1' con vacío. Ver si corresponde tambien a Privado los que con uso_codigo = 1 y descripcion vacía


SELECT *
FROM bootcamp.bronze.transferencias_autos_bronze
WHERE automotor_uso_codigo = 1 and automotor_uso_descripcion IS NULL; --SOLO 5 MAPEAN QUE SI ES NULO LA DESCRIPCION  CUANDO ES 1, CORRESPONDE A Privado

----------------------------------------------------

SELECT DISTINCT titular_tipo_persona
FROM bootcamp.bronze.transferencias_autos_bronze; --Jurídica o Física

SELECT DISTINCT titular_genero, titular_tipo_persona
FROM bootcamp.bronze.transferencias_autos_bronze; --'No aplica' :es para persona Jurídica

SELECT DISTINCT titular_pais_nacimiento
FROM bootcamp.bronze.transferencias_autos_bronze; --128, requiere CASE para mayor prolijidad

SELECT DISTINCT titular_anio_nacimiento
FROM bootcamp.bronze.transferencias_autos_bronze; --149 distintos corregir porque estan raros algunoos años nacimiento que son cualquier cosa 




### Ejercicio 1.4: ¿Cuántas filas por mes?

Cantidad de filas y de días distintos con trámites por cada mes. ¿Algún mes se ve raro (muy chico o muy grande)?

*Pista: `substr(tramite_fecha, 1, 7)` para el mes y `COUNT(DISTINCT tramite_fecha)` para los días.*

In [0]:
%sql
SELECT
  substr(tramite_fecha, 1, 7)       AS mes,
  COUNT(*)                          AS filas,
  COUNT(DISTINCT tramite_fecha)     AS dias_con_tramites
FROM bootcamp.bronze.transferencias_autos_bronze
GROUP BY 1
ORDER BY 1;

-- Hallazgos:
-- 

---
# PARTE 2: Análisis de valores nulos y vacíos
> Ojo: en este dataset la ausencia de dato aparece de **dos formas**: celda vacía (NULL al leer) y el texto literal `No aplica`. Hay que medir las dos.

### Ejercicio 2.1: Contar nulos/vacíos y 'No aplica' en columnas clave

Para `automotor_marca_descripcion, automotor_modelo_descripcion, automotor_tipo_descripcion, automotor_origen, fecha_inscripcion_inicial, titular_tipo_persona, titular_genero, titular_anio_nacimiento, titular_pais_nacimiento, titular_pais_nacimiento_id`: ¿cuántas filas están vacías y cuántas dicen `No aplica`?

*Pista: `COUNT_IF(col IS NULL OR TRIM(col) = '')` para vacíos y `COUNT_IF(UPPER(TRIM(col)) = 'NO APLICA')` para el texto. El `IS NULL` explícito es necesario porque `TRIM(NULL) = ''` da NULL y `COUNT_IF` no lo cuenta.*

In [0]:
%sql
SELECT
  'vacios' AS tipo,
  COUNT_IF(automotor_marca_descripcion IS NULL OR TRIM(automotor_marca_descripcion) = '') AS automotor_marca_descripcion,
  COUNT_IF(automotor_modelo_descripcion IS NULL OR TRIM(automotor_modelo_descripcion) = '') AS automotor_modelo_descripcion,
  COUNT_IF(automotor_tipo_descripcion IS NULL OR TRIM(automotor_tipo_descripcion) = '') AS automotor_tipo_descripcion,
  COUNT_IF(automotor_origen IS NULL OR TRIM(automotor_origen) = '') AS automotor_origen,
  COUNT_IF(fecha_inscripcion_inicial IS NULL OR TRIM(fecha_inscripcion_inicial) = '') AS fecha_inscripcion_inicial,
  COUNT_IF(titular_tipo_persona IS NULL OR TRIM(titular_tipo_persona) = '') AS titular_tipo_persona,
  COUNT_IF(titular_genero IS NULL OR TRIM(titular_genero) = '') AS titular_genero,
  COUNT_IF(titular_anio_nacimiento IS NULL OR TRIM(titular_anio_nacimiento) = '') AS titular_anio_nacimiento,
  COUNT_IF(titular_pais_nacimiento IS NULL OR TRIM(titular_pais_nacimiento) = '') AS titular_pais_nacimiento,
  COUNT_IF(titular_pais_nacimiento_id IS NULL OR TRIM(titular_pais_nacimiento_id) = '') AS titular_pais_nacimiento_id
FROM bootcamp.bronze.transferencias_autos_bronze;

-- Hallazgos:
-- 

In [0]:
%sql
SELECT
  'no_aplica' AS tipo,
  COUNT_IF(UPPER(TRIM(automotor_marca_descripcion)) = 'NO APLICA') AS automotor_marca_descripcion,
  COUNT_IF(UPPER(TRIM(automotor_modelo_descripcion)) = 'NO APLICA') AS automotor_modelo_descripcion,
  COUNT_IF(UPPER(TRIM(automotor_tipo_descripcion)) = 'NO APLICA') AS automotor_tipo_descripcion,
  COUNT_IF(UPPER(TRIM(automotor_origen)) = 'NO APLICA') AS automotor_origen,
  COUNT_IF(UPPER(TRIM(fecha_inscripcion_inicial)) = 'NO APLICA') AS fecha_inscripcion_inicial,
  COUNT_IF(UPPER(TRIM(titular_tipo_persona)) = 'NO APLICA') AS titular_tipo_persona,
  COUNT_IF(UPPER(TRIM(titular_genero)) = 'NO APLICA') AS titular_genero, --acá igual mantener porque es para personas JURÍDICAS EL 'No aplica'
  COUNT_IF(UPPER(TRIM(titular_anio_nacimiento)) = 'NO APLICA') AS titular_anio_nacimiento,
  COUNT_IF(UPPER(TRIM(titular_pais_nacimiento)) = 'NO APLICA') AS titular_pais_nacimiento,
  COUNT_IF(UPPER(TRIM(titular_pais_nacimiento_id)) = 'NO APLICA') AS titular_pais_nacimiento_id
FROM bootcamp.bronze.transferencias_autos_bronze;

-- Hallazgos:
-- 

In [0]:
%sql
SELECT
  COUNT_IF(TRIM(tramite_fecha) <> '' AND TRY_CAST(tramite_fecha AS DATE) IS NULL)                                     AS tramite_fecha_falla,
  COUNT_IF(TRIM(fecha_inscripcion_inicial) <> '' AND TRY_CAST(fecha_inscripcion_inicial AS DATE) IS NULL)             AS inscripcion_falla,
  COUNT_IF(TRIM(automotor_anio_modelo) <> '' AND TRY_CAST(automotor_anio_modelo AS INT) IS NULL)                      AS anio_modelo_falla,
  COUNT_IF(TRIM(titular_anio_nacimiento) <> '' AND TRY_CAST(titular_anio_nacimiento AS INT) IS NULL)                  AS anio_nacimiento_falla,
  COUNT_IF(TRIM(titular_porcentaje_titularidad) <> '' AND TRY_CAST(titular_porcentaje_titularidad AS DOUBLE) IS NULL) AS porcentaje_falla
FROM bootcamp.bronze.transferencias_autos_bronze;

### Ejercicio 2.2: Reporte completo de las 25 columnas

Armá una tabla con **una fila por columna**: total de filas, vacíos, 'No aplica' y sus porcentajes. Ordenala por porcentaje de vacíos.

*Pista: `UNPIVOT` pasa las columnas a filas (todas son texto en Bronze, así que se pueden apilar). Después `GROUP BY columna`.*

In [0]:
%sql
SELECT
  columna,
  COUNT(*)                                                                 AS total,
  COUNT_IF(valor IS NULL OR TRIM(valor) = '')                              AS vacios,
  COUNT_IF(UPPER(TRIM(valor)) = 'NO APLICA')                               AS no_aplica,
  ROUND(COUNT_IF(valor IS NULL OR TRIM(valor) = '') * 100.0 / COUNT(*), 2) AS pct_vacios,
  ROUND(COUNT_IF(UPPER(TRIM(valor)) = 'NO APLICA') * 100.0 / COUNT(*), 2)  AS pct_no_aplica
FROM (SELECT tramite_tipo, tramite_fecha, fecha_inscripcion_inicial, registro_seccional_codigo, registro_seccional_descripcion, registro_seccional_provincia, automotor_origen, automotor_anio_modelo, automotor_tipo_codigo, automotor_tipo_descripcion, automotor_marca_codigo, automotor_marca_descripcion, automotor_modelo_codigo, automotor_modelo_descripcion, automotor_uso_codigo, automotor_uso_descripcion, titular_tipo_persona, titular_domicilio_localidad, titular_domicilio_provincia, titular_genero, titular_anio_nacimiento, titular_pais_nacimiento, titular_porcentaje_titularidad, titular_domicilio_provincia_id, titular_pais_nacimiento_id FROM bootcamp.bronze.transferencias_autos_bronze)
UNPIVOT INCLUDE NULLS (valor FOR columna IN (tramite_tipo, tramite_fecha, fecha_inscripcion_inicial, registro_seccional_codigo, registro_seccional_descripcion, registro_seccional_provincia, automotor_origen, automotor_anio_modelo, automotor_tipo_codigo, automotor_tipo_descripcion, automotor_marca_codigo, automotor_marca_descripcion, automotor_modelo_codigo, automotor_modelo_descripcion, automotor_uso_codigo, automotor_uso_descripcion, titular_tipo_persona, titular_domicilio_localidad, titular_domicilio_provincia, titular_genero, titular_anio_nacimiento, titular_pais_nacimiento, titular_porcentaje_titularidad, titular_domicilio_provincia_id, titular_pais_nacimiento_id))
GROUP BY columna
ORDER BY pct_vacios DESC, pct_no_aplica DESC;

-- Hallazgos:
-- 

### Ejercicio 2.3: Identificar columnas críticas

¿Qué columnas tienen vacíos o 'No aplica' en más del 10% de las filas? ¿Son un problema o es esperable? (Pensá: ¿tiene género una persona jurídica?)

*Pista: reutilizá la consulta anterior envolviéndola en una CTE y filtrá con `WHERE pct_vacios + pct_no_aplica > 10`.*

In [0]:
%sql
WITH reporte AS (
  SELECT
    columna,
    ROUND(COUNT_IF(valor IS NULL OR TRIM(valor) = '') * 100.0 / COUNT(*), 2) AS pct_vacios,
    ROUND(COUNT_IF(UPPER(TRIM(valor)) = 'NO APLICA') * 100.0 / COUNT(*), 2)  AS pct_no_aplica
  FROM (SELECT tramite_tipo, tramite_fecha, fecha_inscripcion_inicial, registro_seccional_codigo, registro_seccional_descripcion, registro_seccional_provincia, automotor_origen, automotor_anio_modelo, automotor_tipo_codigo, automotor_tipo_descripcion, automotor_marca_codigo, automotor_marca_descripcion, automotor_modelo_codigo, automotor_modelo_descripcion, automotor_uso_codigo, automotor_uso_descripcion, titular_tipo_persona, titular_domicilio_localidad, titular_domicilio_provincia, titular_genero, titular_anio_nacimiento, titular_pais_nacimiento, titular_porcentaje_titularidad, titular_domicilio_provincia_id, titular_pais_nacimiento_id FROM bootcamp.bronze.transferencias_autos_bronze)
  UNPIVOT INCLUDE NULLS (valor FOR columna IN (tramite_tipo, tramite_fecha, fecha_inscripcion_inicial, registro_seccional_codigo, registro_seccional_descripcion, registro_seccional_provincia, automotor_origen, automotor_anio_modelo, automotor_tipo_codigo, automotor_tipo_descripcion, automotor_marca_codigo, automotor_marca_descripcion, automotor_modelo_codigo, automotor_modelo_descripcion, automotor_uso_codigo, automotor_uso_descripcion, titular_tipo_persona, titular_domicilio_localidad, titular_domicilio_provincia, titular_genero, titular_anio_nacimiento, titular_pais_nacimiento, titular_porcentaje_titularidad, titular_domicilio_provincia_id, titular_pais_nacimiento_id))
  GROUP BY columna
)
SELECT *
FROM reporte
WHERE pct_vacios + pct_no_aplica > 10 
ORDER BY pct_vacios + pct_no_aplica DESC;

-- Hallazgos:
-- No devuelve problemas del mas del 10%, sin resultado

---
# PARTE 3: Cardinalidad y distribución

### Ejercicio 3.1: Tipos de trámite

¿Qué valores hay en `tramite_tipo`? ¿Cuáles son transferencias a comerciantes habituales (`A COM. HAB.`)? ¿Hay valores truncados (terminan en `DE ...` o cortados)?

*Pista: `GROUP BY tramite_tipo` con conteo y porcentaje (`SUM(COUNT(*)) OVER ()` da el total general).*

In [0]:
%sql
SELECT
  tramite_tipo,
  COUNT(*) AS total,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS porcentaje
FROM bootcamp.bronze.transferencias_autos_bronze
GROUP BY tramite_tipo
ORDER BY total DESC;

-- Hallazgos:
-- 

### Ejercicio 3.2: Origen del automotor

¿Qué proporción de las transferencias es `Nacional`, `Importado` y `Protocolo 21`?

*Pista: `GROUP BY automotor_origen`.*

In [0]:
%sql
SELECT
  automotor_origen,
  COUNT(*) AS total,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS porcentaje
FROM bootcamp.bronze.transferencias_autos_bronze
GROUP BY automotor_origen
ORDER BY total DESC;

-- Hallazgos:
-- 

### Ejercicio 3.3: Tipo de vehículo

¿Cuáles son los 20 tipos de vehículo más frecuentes? ¿Hay el mismo tipo escrito de dos formas (`PUERTAS` / `PTAS`, `SIMPLE` / `SIMPL`)?

*Pista: `GROUP BY automotor_tipo_descripcion` con `LIMIT 20`. Anotá las variantes para normalizarlas en Silver.*

In [0]:
%sql
SELECT
  automotor_tipo_descripcion,
  COUNT(*) AS total,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS porcentaje --denominador es la suma del recuento sobre el TOTAL de la tabla
FROM bootcamp.bronze.transferencias_autos_bronze
GROUP BY automotor_tipo_descripcion
ORDER BY total DESC
LIMIT 20;

-- Hallazgos:
-- sedan 5 puertas, sedan 4 puertas y pickup (en ese orden)

### Ejercicio 3.4: Marcas

¿Cuáles son las 15 marcas con más transferencias? Agrupá por la **descripción** de la marca.

*Pista: `GROUP BY automotor_marca_descripcion` con `LIMIT 15`.*

In [0]:
%sql
SELECT
  automotor_marca_descripcion,
  COUNT(*) AS total,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS porcentaje
FROM bootcamp.bronze.transferencias_autos_bronze
GROUP BY automotor_marca_descripcion
ORDER BY total DESC
LIMIT 15;

-- Hallazgos:
--VOLKSWAGEN
--FORD
--RENAULT
--CHEVROLET
--FIAT
--TOYOTA

### Ejercicio 3.5: ¿El código de marca sirve para agrupar?

En la muestra, una misma marca aparecía con códigos distintos (por ejemplo Toyota con `68` y `130`). ¿Cuántas marcas tienen más de un código? ¿Los códigos distintos se corresponden con el origen (nacional / importado)?

*Pista: `COUNT(DISTINCT automotor_marca_codigo)` por descripción de marca, con `HAVING > 1`, y `COLLECT_SET` para ver los códigos.*

In [0]:
%sql
SELECT
  automotor_marca_descripcion,
  COUNT(DISTINCT automotor_marca_codigo)           AS codigos_distintos,
  COLLECT_SET(automotor_marca_codigo)              AS codigos,
  COLLECT_SET(automotor_origen)                    AS origenes
FROM bootcamp.bronze.transferencias_autos_bronze
GROUP BY automotor_marca_descripcion
HAVING COUNT(DISTINCT automotor_marca_codigo) > 1
ORDER BY codigos_distintos DESC;

-- Hallazgos (si es así, en Silver se agrupa por descripción y no por código):
-- No sirve. Mucho repetido, hay que generar uno nuevo

In [0]:
%sql
SELECT
  automotor_marca_descripcion,
  COUNT(*) AS filas,
  ROUND(SUM(COUNT(*)) OVER (ORDER BY COUNT(*) DESC, automotor_marca_descripcion) * 100.0
        / SUM(COUNT(*)) OVER (), 2) AS pct_acumulado
FROM bootcamp.bronze.transferencias_autos_bronze
GROUP BY automotor_marca_descripcion
ORDER BY filas DESC;

### Ejercicio 3.6: Provincia del registro

¿En qué provincias se registran más transferencias? ¿Cuánto pesa la Ciudad de Buenos Aires y Buenos Aires?

*Pista: `GROUP BY registro_seccional_provincia`.*

In [0]:
%sql
SELECT
  registro_seccional_provincia,
  COUNT(*) AS total,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS porcentaje
FROM bootcamp.bronze.transferencias_autos_bronze
GROUP BY registro_seccional_provincia
ORDER BY total DESC
LIMIT 25;

-- Hallazgos:
-- Similar la cantidad de transferencias por la cantidad de población en cada provincia

### Ejercicio 3.7: Persona física o jurídica

¿Qué proporción de los titulares son personas jurídicas?

*Pista: `GROUP BY titular_tipo_persona`.*

In [0]:
%sql
SELECT
  titular_tipo_persona,
  COUNT(*) AS total,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS porcentaje
FROM bootcamp.bronze.transferencias_autos_bronze
GROUP BY titular_tipo_persona
ORDER BY total DESC;

-- Hallazgos:
-- 93% personas humanas

### Ejercicio 3.8: Uso del automotor

¿Qué valores tiene `automotor_uso_descripcion`? ¿Aporta información o es siempre lo mismo?

*Pista: `GROUP BY automotor_uso_descripcion`. Si es una sola categoría, la columna no sirve para analizar.*

In [0]:
%sql
SELECT
  automotor_uso_descripcion,
  COUNT(*) AS total,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS porcentaje
FROM bootcamp.bronze.transferencias_autos_bronze
GROUP BY automotor_uso_descripcion
ORDER BY total DESC;

-- Hallazgos:
-- 

---
# PARTE 4: Estadísticas descriptivas de variables numéricas y fechas

> **Nota sobre formato de datos:** en Bronze todo es texto. Para calcular estadísticas primero creamos una vista temporal `transf_clean` que convierte los tipos con `TRY_CAST` (si el valor no se puede convertir queda `NULL`, no rompe) y agrega columnas derivadas.

> **Ojo:** la vista temporal vive solo mientras el cómputo está encendido. Si se apaga, volver a ejecutar esta celda.

In [0]:
%sql
-- Vista temporal con tipos correctos y columnas derivadas
CREATE OR REPLACE TEMPORARY VIEW transf_clean AS
SELECT
  regexp_extract(_archivo_origen, '([0-9]{6})', 1)                  AS periodo_archivo,
  tramite_tipo,
  TRY_CAST(tramite_fecha AS DATE)                                   AS tramite_fecha,
  TRUNC(TRY_CAST(tramite_fecha AS DATE), 'MM')                      AS mes,
  TRY_CAST(fecha_inscripcion_inicial AS DATE)                       AS fecha_inscripcion_inicial,
  registro_seccional_codigo,
  registro_seccional_descripcion,
  registro_seccional_provincia,
  automotor_origen,
  COALESCE(TRY_CAST(automotor_anio_modelo AS INT), NULL)                            AS automotor_anio_modelo,
  automotor_tipo_codigo,
  -- normalización de variantes vistas en la muestra (completar con lo que encuentres en 3.3)
  regexp_replace(
  regexp_replace(UPPER(TRIM(automotor_tipo_descripcion)), ' PTAS$', ' PUERTAS'),
  'PICK UP', 'PICK-UP')                                             AS automotor_tipo_descripcion,
  automotor_marca_codigo,
  TRIM(automotor_marca_descripcion)                                 AS automotor_marca_descripcion,
  automotor_modelo_codigo,
  TRIM(automotor_modelo_descripcion)                                AS automotor_modelo_descripcion,
  -- primera palabra del modelo: agrupa versiones (limitación: COROLLA y COROLLA CROSS quedan juntos)
  split(TRIM(automotor_modelo_descripcion), ' ')[0]                 AS modelo_base,
  automotor_uso_descripcion,
  titular_tipo_persona,
  titular_domicilio_provincia,
  titular_genero             AS titular_genero,
  TRY_CAST(titular_anio_nacimiento AS INT)                          AS titular_anio_nacimiento,
  NULLIF(NULLIF(TRIM(titular_pais_nacimiento), ''), 'No aplica')    AS titular_pais_nacimiento,
  TRY_CAST(titular_porcentaje_titularidad AS DOUBLE)                AS titular_porcentaje_titularidad,
  -- peso de la fila: un auto con 2 titulares al 50% cuenta 0,5 + 0,5 = 1 transferencia
  COALESCE(TRY_CAST(titular_porcentaje_titularidad AS DOUBLE), 100.0) / 100.0 AS peso,
  -- hipótesis: 'A COM. HAB.' = transferencia a comerciante habitual (concesionaria). A confirmar con la documentación
  COALESCE(tramite_tipo LIKE '%COM. HAB%', FALSE)                   AS a_comerciante_habitual,
  ROUND(DATEDIFF(TRY_CAST(tramite_fecha AS DATE),
                 TRY_CAST(fecha_inscripcion_inicial AS DATE)) / 365.25, 1) AS antiguedad_anios,
  CASE WHEN titular_tipo_persona LIKE 'F%'
       THEN YEAR(TRY_CAST(tramite_fecha AS DATE)) - TRY_CAST(titular_anio_nacimiento AS INT)
  END                                                               AS edad_titular
FROM bootcamp.bronze.transferencias_autos_bronze;
--)
--select count(*) from prueba;

-- Verificación rápida:

SELECT COUNT(*) FROM transf_clean; --LIMIT 10;

In [0]:
%sql
--análisis respecto a trámite tipo:

SELECT tramite_tipo, COUNT(*) AS filas
FROM bootcamp.bronze.transferencias_autos_bronze
GROUP BY tramite_tipo;

--correr la vista temporal, luego ejecutar esta: 

SELECT
  a_comerciante_habitual,
  titular_tipo_persona,
  COUNT(*)                                       AS filas,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY a_comerciante_habitual), 1) AS pct_del_grupo
FROM transf_clean
GROUP BY a_comerciante_habitual, titular_tipo_persona
ORDER BY a_comerciante_habitual DESC, filas DESC;

SELECT DISTINCT automotor_tipo_descripcion FROM transf_clean;

SELECT automotor_tipo_descripcion, COUNT(*) AS total,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS porcentaje
FROM transf_clean
GROUP BY automotor_tipo_descripcion
ORDER BY total DESC
LIMIT 30;

--Pensar en un CASE capa silver con nueva columna de: categoria:auto

![image_1795185867781.png](./image_1795185867781.png "image_1795185867781.png")

### Ejercicio 4.1: Antigüedad del auto al transferirse

**Objetivo:** ¿qué antigüedad tienen los autos que se transfieren?

**Pregunta:** calculá cantidad, mínimo, máximo, promedio, mediana, percentil 25 y 75 de `antiguedad_anios`, en total y por `automotor_origen`. Excluí antigüedades negativas.

*Pista: `PERCENTILE(col, 0.5)` para la mediana. `UNION ALL` para mostrar el total y los grupos juntos.*

In [0]:
%sql
--recordar que antiguedad anios es un campo calculado en la temporal
SELECT
  'total' AS grupo,
  COUNT(*) AS cantidad,
  MIN(antiguedad_anios) AS minimo,
  MAX(antiguedad_anios) AS maximo,
  ROUND(AVG(antiguedad_anios), 1) AS promedio,
  ROUND(PERCENTILE(antiguedad_anios, 0.25), 1) AS p25,
  ROUND(PERCENTILE(antiguedad_anios, 0.5), 1)  AS mediana,
  ROUND(PERCENTILE(antiguedad_anios, 0.75), 1) AS p75
FROM transf_clean
WHERE antiguedad_anios >= 0
UNION ALL
SELECT
  automotor_origen,
  COUNT(*),
  MIN(antiguedad_anios),
  MAX(antiguedad_anios),
  ROUND(AVG(antiguedad_anios), 1),
  ROUND(PERCENTILE(antiguedad_anios, 0.25), 1),
  ROUND(PERCENTILE(antiguedad_anios, 0.5), 1),
  ROUND(PERCENTILE(antiguedad_anios, 0.75), 1)
FROM transf_clean
WHERE antiguedad_anios >= 0
GROUP BY automotor_origen;

-- Hallazgos:
-- 

### Ejercicio 4.2: Año modelo

**Pregunta:** ¿cuál es el rango de `automotor_anio_modelo`? ¿Cuáles son los 15 años modelo con más transferencias? ¿Hay años imposibles (anteriores a 1950 o más de un año posterior al trámite)?

*Pista: `GROUP BY automotor_anio_modelo ORDER BY total DESC LIMIT 15` y un conteo aparte con `COUNT_IF` para los años imposibles.*

In [0]:
%sql
SELECT
  automotor_anio_modelo,
  COUNT(*) AS total,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS porcentaje
FROM transf_clean
GROUP BY automotor_anio_modelo
ORDER BY total DESC

-- Hallazgos:
-- No considerar los autos con año modelo: mayores a 2026. No tiene sentido. 

SELECT COUNT(*) FROM transf_clean WHERE automotor_anio_modelo is NULL; --tenemos 7078 casos de automotores sin año de modelo. Dejarlos como está porque aporta info valiosa en las otras columnas (esta bien el COALESCE de )

In [0]:
%sql
SELECT
  MIN(automotor_anio_modelo)                                        AS anio_min,
  MAX(automotor_anio_modelo)                                        AS anio_max,
  COUNT_IF(automotor_anio_modelo IS NULL)                           AS anio_nulo,
  COUNT_IF(automotor_anio_modelo < 1950)                            AS anterior_1950,
  COUNT_IF(automotor_anio_modelo > YEAR(tramite_fecha) + 1)         AS posterior_al_tramite_mas_1
FROM transf_clean;

-- Hallazgos:
-- 

### Ejercicio 4.3: Edad de los titulares (personas físicas)

**Pregunta:** ¿cuál es la edad mínima, máxima, promedio y mediana de los titulares que son personas físicas? ¿Hay edades imposibles (menores de 18 o mayores de 100)? Y en las personas **jurídicas**, ¿qué contiene `titular_anio_nacimiento` (¿un año de constitución?).

*Pista: usá `edad_titular` (ya calculada en la vista, solo para personas físicas). Para las jurídicas mirá el rango de `titular_anio_nacimiento`.*

In [0]:
%sql
SELECT
  COUNT(*)                                         AS personas_fisicas,
  MIN(edad_titular)                                AS edad_min,
  MAX(edad_titular)                                AS edad_max,
  ROUND(AVG(edad_titular), 1)                      AS edad_promedio,
  ROUND(PERCENTILE(edad_titular, 0.5), 1)          AS edad_mediana,
  COUNT_IF(edad_titular < 18 OR edad_titular > 110) AS edades_imposibles
FROM transf_clean
WHERE titular_tipo_persona LIKE 'F%'; 

-- Hallazgos:
-- Cambié el limite superiorr de edad a 110... puede pasar. Tenemos 469 casos imposibles. Filtrarlos

In [0]:
%sql
SELECT
  MIN(titular_anio_nacimiento) AS anio_min,
  MAX(titular_anio_nacimiento) AS anio_max,
  COUNT(*)                     AS personas_juridicas
FROM transf_clean
WHERE titular_tipo_persona LIKE 'J%';

-- Hallazgos (¿parece un año de constitución?):
-- Dejar como está. Para personas juridicas no tiene sentido hacer un análisis por titular_anio_nacimiento.
-- Encima hay casos de anio_max > 2026 con persona juridica, es imposible saber porque pusieron esa fecha
-- Poner todas en caso de personas jurídicas, NULL el campo: titular_anio_nacimiento

SELECT * FROM transf_clean WHERE titular_tipo_persona LIKE 'J%' AND titular_anio_nacimiento > 2026

### Ejercicio 4.4: Porcentaje de titularidad

**Pregunta:** ¿qué valores toma `titular_porcentaje_titularidad`? ¿Qué proporción de filas tiene menos de 100%?

*Pista: `GROUP BY titular_porcentaje_titularidad ORDER BY total DESC LIMIT 10`.*

In [0]:
%sql
SELECT
  titular_porcentaje_titularidad,
  COUNT(*) AS total,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS porcentaje
FROM transf_clean
GROUP BY titular_porcentaje_titularidad
ORDER BY total DESC;

-- Hallazgos:
-- 

---
# PARTE 5: Detección de problemas de calidad

In [0]:
%sql
DESCRIBE transf_clean;

### Ejercicio 5.1: Reporte de calidad usando CTEs

Creá un reporte con: total de registros y cantidad de filas con: fecha de trámite inválida, fecha de inscripción inicial inválida, inscripción **posterior** al trámite, año modelo inválido, edad de titular imposible (físicas), marca vacía, modelo vacío, tipo de trámite vacío, porcentaje de titularidad fuera de (0, 100]. Mostrá también el porcentaje de cada uno.

*Pista: `COUNT_IF(...)` con `IS NULL OR` explícito en cada condición (un `NULL` en una comparación no se cuenta).*

In [0]:
%sql
WITH total AS (
  SELECT COUNT(*) AS gran_total FROM transf_clean
),
metricas AS (
  SELECT
    COUNT_IF(tramite_fecha IS NULL)                                                         AS fecha_tramite_inv,
    COUNT_IF(fecha_inscripcion_inicial IS NULL)                                             AS fecha_inscripcion_inv,
    COUNT_IF(fecha_inscripcion_inicial > tramite_fecha)                                     AS inscripcion_posterior,
    COUNT_IF(automotor_anio_modelo IS NULL OR automotor_anio_modelo < 1900
             OR automotor_anio_modelo > YEAR(tramite_fecha) + 1)                            AS anio_modelo_inv,
    COUNT_IF(titular_tipo_persona LIKE 'F' || '%'
             AND (edad_titular IS NULL OR edad_titular < 18 OR edad_titular > 110))         AS edad_titular_inv,
    COUNT_IF(automotor_marca_descripcion IS NULL OR automotor_marca_descripcion = '')       AS marca_vacia,
    COUNT_IF(automotor_modelo_descripcion IS NULL OR automotor_modelo_descripcion = '')     AS modelo_vacio,
    COUNT_IF(tramite_tipo IS NULL OR TRIM(tramite_tipo) = '')                               AS tramite_tipo_vacio,
    COUNT_IF(titular_porcentaje_titularidad IS NULL
             OR titular_porcentaje_titularidad <= 0
             OR titular_porcentaje_titularidad > 100)                                       AS pct_titularidad_inv
  FROM transf_clean
)
SELECT
  t.gran_total,
  m.*,
  ROUND(m.inscripcion_posterior * 100.0 / t.gran_total, 4) AS pct_inscripcion_posterior,
  ROUND(m.anio_modelo_inv * 100.0 / t.gran_total, 4)       AS pct_anio_modelo_inv,
  ROUND(m.edad_titular_inv * 100.0 / t.gran_total, 4)      AS pct_edad_titular_inv
FROM total t
CROSS JOIN metricas m;

-- Hallazgos:
-- Recordar los casos de personas jkuridicas, no poner el titular_anio_naicmiento

### Ejercicio 5.2: ¿Una fila es un auto o un titular? (copropiedad)

No hay patente, así que no podemos identificar un auto directamente. Pero si los porcentajes de titularidad de todos los titulares de **una misma transferencia** suman 100, entonces `SUM(titular_porcentaje_titularidad) / 100` es la cantidad real de transferencias.

**Pregunta:** ¿cuántas filas hay? ¿Cuántas transferencias reales estima `SUM(pct) / 100`? ¿Cuánto cambia?

*Pista: comparé `COUNT(*)` con `SUM(peso)` (la columna `peso` de la vista ya es el porcentaje dividido 100).*

In [0]:
%sql
SELECT
  COUNT(*)                                          AS filas,
  ROUND(SUM(peso))                                  AS transferencias_estimadas,
  COUNT(*) - ROUND(SUM(peso))                       AS filas_de_mas,
  ROUND((COUNT(*) - SUM(peso)) * 100.0 / COUNT(*), 2) AS pct_de_mas
FROM transf_clean;

-- Hallazgos (si la diferencia es chica, contar filas o contar transferencias casi no cambia los rankings):
-- 

In [0]:
%sql
SELECT periodo_archivo, SUM(peso) AS transferencias_estimadas
FROM transf_clean
GROUP BY periodo_archivo
ORDER BY periodo_archivo;

In [0]:
%sql
SELECT COUNT(*) FROM transf_clean WHERE peso <> 1;

In [0]:
%sql
--Analizando la coporpoiedad de transfrencias: 
SELECT
  automotor_marca_descripcion                                              AS marca,
  modelo_base,
  COUNT(*)                                                                 AS filas,
  COUNT_IF(titular_porcentaje_titularidad = 100)                           AS filas_al_100,
  COUNT_IF(titular_porcentaje_titularidad < 100)                           AS filas_menos_de_100,
  ROUND(COUNT_IF(titular_porcentaje_titularidad < 100) * 100.0 / COUNT(*), 1) AS pct_menos_de_100
FROM transf_clean
GROUP BY automotor_marca_descripcion, modelo_base
ORDER BY filas DESC
LIMIT 15;

-- Hallazgos: la copropiedad (titularidad < 100%) afecta al 4,9% de las filas, pero varía por modelo
-- (de 3,1% en Corsa a 7,0% en Corolla). El ranking de los 15 modelos principales no cambia al filtrar
-- al 100%, pero filtrar subestimaría a los modelos con más copropiedad. Decisión: conservar todas las
-- filas con su peso (SUM(peso)) y no filtrar en Silver.

### Ejercicio 5.3: Detectar duplicados

¿Hay filas idénticas en todas las columnas relevantes? Agrupá por todas las columnas de trámite, auto y titular y encontrá los grupos que aparecen más de una vez. Mostrá: cantidad de grupos duplicados, registros duplicados y registros «extra».

> **Ojo con la interpretación:** sin patente, dos filas iguales pueden ser el mismo registro cargado dos veces **o** dos autos distintos iguales vendidos el mismo día a la misma persona (por ejemplo una flota). No se puede borrar a ciegas: hay que decidir con los hallazgos.

*Pista: `GROUP BY ALL HAVING COUNT(*) > 1`, y luego sumá los resultados con una CTE.*

In [0]:
%sql
WITH grupos AS (
  SELECT
    * EXCEPT (_ingesta_ts, _rescued_data),
    COUNT(*) AS veces
  FROM bootcamp.bronze.transferencias_autos_bronze
  GROUP BY ALL
  HAVING COUNT(*) > 1
)
SELECT
  COUNT(*)         AS grupos_duplicados,
  SUM(veces)       AS registros_duplicados,
  SUM(veces - 1)   AS registros_extra
FROM grupos; 

-- Hallazgos: con las 25 columnas hay 1.704 grupos repetidos (2.536 filas extra, 0,18% de la tabla).
-- Con claves más laxas el número se infla (8 columnas: 15.757) por coincidencias de modelos populares.
-- Los grupos más grandes son personas jurídicas (flotas). Decisión: no borrar; marcar como
-- No es duplicado porque corresponde a compra de flota de empresas.

In [0]:
%sql
WITH grupos AS (
  SELECT
    * EXCEPT (_ingesta_ts, _rescued_data),
    COUNT(*) AS veces
  FROM bootcamp.bronze.transferencias_autos_bronze
  GROUP BY ALL
  HAVING COUNT(*) > 1
)
SELECT
  titular_tipo_persona,
  titular_porcentaje_titularidad,
  CASE 
  WHEN veces = 2 THEN '2 filas' 
  WHEN veces <= 5 THEN '3 a 5 filas' 
  ELSE '6 o más filas' 
  END AS tamano_grupo,
  COUNT(*)          AS grupos,
  SUM(veces - 1)    AS filas_extra
FROM grupos
GROUP BY titular_tipo_persona, titular_porcentaje_titularidad, 3
ORDER BY filas_extra DESC;

### Ejercicio 5.4: Ver ejemplos de duplicados

Mostrá los 10 grupos con más repeticiones. ¿Parecen flotas de concesionarias o personas jurídicas (varios autos iguales a la vez)?

*Pista: la misma agrupación, ordenada por `veces DESC` con `LIMIT 10`. Fijate en `titular_tipo_persona` y `tramite_tipo`.*

In [0]:
%sql
SELECT
  *,
  COUNT(*) AS veces
FROM transf_clean
GROUP BY ALL
HAVING COUNT(*) > 1
ORDER BY veces DESC;



### Ejercicio 5.5: Detectar outliers en antigüedad

Calculá los percentiles 1 y 99 de `antiguedad_anios` por origen y listá los autos más viejos. ¿Un auto de 1980 transferido en 2026 es un error o un caso real (clásico, camión)?

*Pista: una CTE con `PERCENTILE(antiguedad_anios, 0.01)` y `0.99` por `automotor_origen`, y luego un `JOIN` para contar cuántos quedan fuera.*

**Ojo con la interpetación de antiguedad anios**: Entiendo que se interpreta como la edad que tiene el automovil al momento de transfeir, no la edad actual

In [0]:
%sql
WITH limites AS (
  SELECT
    automotor_origen,
    PERCENTILE(antiguedad_anios, 0.01) AS p1,
    PERCENTILE(antiguedad_anios, 0.99) AS p99
  FROM transf_clean
  WHERE antiguedad_anios >= 0
  GROUP BY automotor_origen
)
SELECT
  t.automotor_origen,
  l.p1,
  l.p99,
  COUNT_IF(t.antiguedad_anios < l.p1)  AS bajo_p1,
  COUNT_IF(t.antiguedad_anios > l.p99) AS sobre_p99,
  MAX(t.antiguedad_anios)              AS maxima_antiguedad
FROM transf_clean t
JOIN limites l ON t.automotor_origen = l.automotor_origen
GROUP BY t.automotor_origen, l.p1, l.p99;

-- Hallazgos:
-- No borrar ningun dato por antiuedad por debajo del percentil p1, o p99, porque osn casos reales

In [0]:
%sql
SELECT
  automotor_marca_descripcion, automotor_modelo_descripcion, automotor_tipo_descripcion,
  automotor_anio_modelo, fecha_inscripcion_inicial, antiguedad_anios
FROM transf_clean
ORDER BY antiguedad_anios DESC
LIMIT 10;

-- Hallazgos:
-- 

### Ejercicio 5.6: ¿Los registros de un archivo corresponden a su mes?

Contá, por archivo, cuántas filas tienen una `tramite_fecha` de **otro mes** distinto al del nombre del archivo. Si hay, ¿son pocas (errores de carga) o muchas (archivo mal armado)?

*Pista: comparar `periodo_archivo` con `date_format(tramite_fecha, 'yyyyMM')`.*

In [0]:
%sql
SELECT
  periodo_archivo,
  COUNT(*)                                                         AS filas,
  COUNT_IF(date_format(tramite_fecha, 'yyyyMM') <> periodo_archivo) AS filas_de_otro_mes
FROM transf_clean
GROUP BY periodo_archivo
ORDER BY periodo_archivo;

-- Hallazgos:
-- 

### Ejercicio 5.7: Reglas candidatas para Silver

A partir de todos los hallazgos de las Partes 2 a 5, completá la tabla (borrador, ajustalo con lo que encuentres):

| Problema encontrado | Regla en Silver | Decisión |
|---|---|---|
| `No aplica` en género y país de nacimiento (personas jurídicas) | Dejar No aplica para diferenciar métricas de personas jurídicas | limpiar, pero no transofrmar |
| Mismo tipo de vehículo escrito de dos formas (`PTAS` / `PUERTAS`, `SIMPL` / `SIMPLE`) | normalizar la descripción | transformar |
| Mismo nombre de marca con códigos distintos | agrupar por descripción, no por código | transformar |
| Filas con varios titulares (porcentaje < 100) | usar `peso` o `SUM(pct)/100` para contar transferencias | usar peso |
| Duplicados exactos (¿flotas o doble carga?) | no borrar sin evidencia; analizar primero | no borrar mada porque los 'duplicados' suelen ser personas jurídicas con compra de flotas de autos iguales (de igual marca - modelo) |
| `titular_anio_nacimiento` en jurídicas | poner ese campo NULL para personas jurídicas | NULL cuando titular_tipo_persona = 'Jurídica' |
| Fechas o años imposibles | marcar o descartar | Autos con fecha de anio_modelo > CURRENT_YEAR poner flag como 9999 podría ser |

### Definición de granularidad (MUY CLAVE PARA SILVER Y PRÓXIMAS CAPAS): 
Cada fila es una transferencia con titular o es un titular con transferencia? 


In [0]:
%sql
WITH g AS (
  SELECT
    tramite_fecha, registro_seccional_codigo, automotor_marca_descripcion,
    automotor_modelo_codigo, automotor_anio_modelo, fecha_inscripcion_inicial,
    COUNT(*)                            AS filas,
    SUM(titular_porcentaje_titularidad) AS suma_pct
  FROM transf_clean
  GROUP BY ALL
  HAVING MIN(titular_porcentaje_titularidad) < 100
)
SELECT
  CASE WHEN ABS(suma_pct - 100) < 0.5 THEN 'suma 100'
       WHEN suma_pct < 100            THEN 'suma menos de 100'
       ELSE 'suma más de 100' END     AS resultado,
  COUNT(*)                            AS grupos,
  SUM(filas)                          AS filas
FROM g
GROUP BY 1
ORDER BY grupos DESC;

Se concluye que cada fila es una TRANSFERENCIA. Cada transferencia corresponde a un titular

- Una fila es una titularidad adquirida, que es lo más cercano a "una transferencia" que tiene el dataset. COUNT(*) pasa a ser la medida principal y puede llamarse transferencias registradas (con la salvedad de que una compra conjunta de dos personas son dos filas, algo raro: 606 filas, 0,04% de la tabla).

---
# PARTE 6: Análisis avanzado con Window Functions
**Pregunta de negocio:** ¿qué autos se transfieren más en cada mes y qué tan estable es ese ranking?
> Contamos **transferencias estimadas** (`SUM(peso)`) para no duplicar autos con varios titulares. Podés compararlo con `COUNT(*)`.

### Ejercicio 6.1: Evolución mensual con variación respecto del mes anterior

Mostrá por mes: filas, transferencias estimadas, transferencias por día con trámites, y la variación porcentual contra el mes anterior.

*Pista: `LAG(columna) OVER (ORDER BY mes)` trae el valor del mes anterior. Los meses tienen distinta cantidad de días: por eso conviene mirar también el promedio por día.*

In [0]:
%sql
WITH mensual AS (
  SELECT
    mes,
    COUNT(*)                         AS transferencias_est,
    COUNT(DISTINCT tramite_fecha)    AS dias_con_tramites
  FROM transf_clean
  GROUP BY mes
)
SELECT
  mes,
  transferencias_est,
  dias_con_tramites,
  ROUND(transferencias_est / dias_con_tramites)                                 AS por_dia,
  LAG(transferencias_est) OVER (ORDER BY mes)                                   AS mes_anterior,
  ROUND((transferencias_est - LAG(transferencias_est) OVER (ORDER BY mes)) * 100.0
        / LAG(transferencias_est) OVER (ORDER BY mes), 2)                       AS var_pct_vs_mes_anterior
FROM mensual
ORDER BY mes;

-- Hallazgos:
-- 

### Ejercicio 6.2: Ranking de marcas por mes

Para cada mes, las 10 marcas con más transferencias y su posición.

*Pista: `RANK() OVER (PARTITION BY mes ORDER BY transferencias DESC)` y luego filtrá `rk <= 10` en una consulta exterior.*

In [0]:
%sql
WITH conteo AS (
  SELECT
    mes,
    automotor_marca_descripcion AS marca,
    COUNT(*)            AS transferencias_est
  FROM transf_clean
  GROUP BY mes, automotor_marca_descripcion
),
ranking AS (
  SELECT
    *,
    RANK() OVER (PARTITION BY mes ORDER BY transferencias_est DESC) AS rk
  FROM conteo
)
SELECT mes, rk, marca, transferencias_est
FROM ranking
WHERE rk <= 10
ORDER BY mes, rk;

-- Hallazgos:
-- Por cada mes, un ranking del 1 al 10 con la cantidad de transferencias por marca. 
--Predomina bastante volsvagen (sospescho de la amarok)

### Ejercicio 6.3: Top 10 de modelos del último mes y su recorrido

Tomá los 10 modelos (marca + `modelo_base`) con más transferencias en el **último mes** y mostrá su cantidad y posición en **todos** los meses. ¿Son estables o saltan de posición?

*Pista: una CTE de ranking por mes, otra con el top 10 del último mes (`MAX(mes)`), y un `JOIN` para traer sus meses.*

In [0]:
%sql
WITH n AS (
  SELECT
    automotor_marca_descripcion AS marca,
    modelo_base,
    mes,
    COUNT(*)            AS transferencias_est
  FROM transf_clean
  GROUP BY automotor_marca_descripcion, modelo_base, mes
),
rk AS (
  SELECT
    *,
    RANK() OVER (PARTITION BY mes ORDER BY transferencias_est DESC) AS posicion 
    --necesito para el ranking para la agrupación anterior
  FROM n
),
top_ult_mes AS (
  SELECT marca, modelo_base
  FROM rk
  WHERE mes = (SELECT MAX(mes) FROM n) AND posicion <= 10
)
SELECT
  rk.marca,
  rk.modelo_base,
  rk.mes,
  rk.transferencias_est,
  rk.posicion
FROM rk
JOIN top_ult_mes t ON rk.marca = t.marca AND rk.modelo_base = t.modelo_base
ORDER BY rk.marca, rk.modelo_base, rk.mes;

-- Hallazgos:
-- Solo muestra a los modelos que están en el top 10 de septiembre. Si otro modelo estaba en el top 10 en meses --anteriores y se cayó, no aparece. Es la historia de "los que están arriba hoy", no de quién entró y quién salió.
--Los conteos absolutos mezclan dos cosas: el modelo y el mes (más o menos días, más o menos mercado). La posición --es más confiable para comparar entre meses, porque ya descuenta el efecto del mes.

### Ejercicio 6.4: Promedio móvil de 3 meses

Para los mismos modelos, calculá el promedio móvil de 3 meses y compará cada mes con ese promedio. Un modelo que supera de forma sostenida su promedio está en alza.

*Pista: `AVG(transferencias_est) OVER (PARTITION BY marca, modelo_base ORDER BY mes ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)`.*

In [0]:
%sql
WITH n AS (
  SELECT
    automotor_marca_descripcion AS marca,
    modelo_base,
    mes,
    ROUND(SUM(peso))            AS transferencias_est
  FROM transf_clean
  GROUP BY automotor_marca_descripcion, modelo_base, mes
),
top_ult_mes AS (
  SELECT marca, modelo_base
  FROM n
  WHERE mes = (SELECT MAX(mes) FROM n)
  ORDER BY transferencias_est DESC
  LIMIT 10
)
SELECT
  n.marca,
  n.modelo_base,
  n.mes,
  n.transferencias_est,
  ROUND(AVG(n.transferencias_est) OVER (
    PARTITION BY n.marca, n.modelo_base
    ORDER BY n.mes
    ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 1) AS promedio_movil_3m
FROM n
JOIN top_ult_mes t ON n.marca = t.marca AND n.modelo_base = t.modelo_base
ORDER BY n.marca, n.modelo_base, n.mes;

-- Hallazgos:
-- El promedio movil 3m, toma del mes actual + los 2 anteriores de la fila

### Ejercicio 6.5: Origen del auto mes a mes

¿Cambia la mezcla de nacionales / importados / `Protocolo 21` a lo largo de los meses?

*Pista: `SUM(SUM(peso)) OVER (PARTITION BY mes)` da el total del mes para calcular el porcentaje.*

In [0]:
%sql
SELECT
  mes,
  automotor_origen,
  ROUND(SUM(peso))                                              AS transferencias_est,
  ROUND(SUM(peso) * 100.0 / SUM(SUM(peso)) OVER (PARTITION BY mes), 2) AS pct_del_mes
FROM transf_clean
GROUP BY mes, automotor_origen
ORDER BY mes, transferencias_est DESC;

-- Hallazgos:
-- 

### Ejercicio 6.6: Comerciantes habituales (caso concesionarias)

1. ¿Qué proporción de las transferencias de cada mes es a comerciantes habituales?
2. ¿Los autos que compran los comerciantes son más nuevos que los de los particulares?
3. ¿Qué modelos compran más los comerciantes en el último mes?

> Recordá que `a_comerciante_habitual` es una **hipótesis** (`tramite_tipo` contiene `COM. HAB.`). Confirmar el significado en la documentación del dataset.

In [0]:
%sql
SELECT
  mes,
  ROUND(SUM(peso))                                                        AS total_est,
  ROUND(SUM(CASE WHEN a_comerciante_habitual THEN peso END))              AS a_comerciantes,
  ROUND(SUM(CASE WHEN a_comerciante_habitual THEN peso END) * 100.0 / SUM(peso), 2) AS pct_comerciantes
FROM transf_clean
GROUP BY mes
ORDER BY mes;

-- Hallazgos:
-- 

In [0]:
%sql
SELECT
  a_comerciante_habitual, --sospecho de como se creó esta columna,
  COUNT(*)                                       AS filas,
  ROUND(AVG(antiguedad_anios), 1)                AS antiguedad_promedio,
  ROUND(PERCENTILE(antiguedad_anios, 0.5), 1)    AS antiguedad_mediana
FROM transf_clean
WHERE antiguedad_anios >= 0
GROUP BY a_comerciante_habitual;

-- Hallazgos:
-- Revisando como se creó 'a_comerciante_habitual', encuentro otra posible reacion de cmapo: 

-- CASE
--   -- tramite_tipo "... A COM. HAB.": el comerciante recibe el auto (lo compra, entra a su stock)
--   WHEN tramite_tipo LIKE '% A COM. HAB%'  THEN 'COMERCIANTE_COMPRA'
--   -- tramite_tipo "... DE COM. HAB.": el comerciante entrega el auto (lo vende a un cliente)
--   WHEN tramite_tipo LIKE '% DE COM. HAB%' THEN 'COMERCIANTE_VENDE'
--   -- cualquier otro trámite: sin comerciante habitualista en la operación
--   ELSE 'SIN_COMERCIANTE'
-- END AS rol_comerciante



In [0]:
%sql
WITH base AS (
  SELECT
    CASE --case mas explicito para rol_comerciante
  -- tramite_tipo "... A COM. HAB.": el comerciante recibe el auto (lo compra, entra a su stock)
  WHEN tramite_tipo LIKE '% A COM. HAB%'  THEN 'COMERCIANTE_COMPRA'
  -- tramite_tipo "... DE COM. HAB.": el comerciante entrega el auto (lo vende a un cliente)
  WHEN tramite_tipo LIKE '% DE COM. HAB%' THEN 'COMERCIANTE_VENDE'
  -- cualquier otro trámite: sin comerciante habitualista en la operación
  ELSE 'SIN_COMERCIANTE'
END AS rol_comerciante,
    titular_tipo_persona
  FROM bootcamp.bronze.transferencias_autos_bronze
),
conteo AS (
  SELECT rol_comerciante, titular_tipo_persona, COUNT(*) AS filas
  FROM base
  GROUP BY rol_comerciante, titular_tipo_persona
)
SELECT
  rol_comerciante,
  titular_tipo_persona,
  filas,
  ROUND(filas * 100.0 / SUM(filas) OVER (PARTITION BY rol_comerciante), 1) AS pct_del_grupo
FROM conteo
ORDER BY rol_comerciante, filas DESC;

In [0]:
%sql
SELECT
  automotor_marca_descripcion AS marca,
  modelo_base,
  COUNT(*)            AS transferencias
FROM transf_clean
WHERE a_comerciante_habitual
  AND mes = (SELECT MAX(mes) FROM transf_clean)
GROUP BY automotor_marca_descripcion, modelo_base
ORDER BY transferencias DESC
LIMIT 15;

-- Hallazgos:
-- 

### Ejercicio 6.7: Calendario del mes

¿Hay días de la semana o del mes con más transferencias? (Con un solo año los datos de estacionalidad son limitados: lo que veas acá es orientativo.)

*Pista: `DAYOFWEEK(fecha)` (1 = domingo) y `DAY(fecha)`.*

In [0]:
%sql
SELECT
  DAYOFWEEK(tramite_fecha)            AS dow,
  DATE_FORMAT(tramite_fecha, 'EEEE')  AS dia,
  ROUND(SUM(peso))                    AS transferencias_est
FROM transf_clean
GROUP BY DAYOFWEEK(tramite_fecha), DATE_FORMAT(tramite_fecha, 'EEEE')
ORDER BY dow;

-- Hallazgos:
-- 

In [0]:
%sql
SELECT
  DAY(tramite_fecha)  AS dia_del_mes,
  COUNT(*)   AS transferencias_est
FROM transf_clean
GROUP BY DAY(tramite_fecha)
ORDER BY dia_del_mes;

-- Hallazgos (¿picos a fin de mes?):
-- No se observa un patrón claro de día del mes. Descartar

### Ejercicio 6.8: ¿Los modelos que más se transfieren están en alza o en baja?

Comparé la suma de los **primeros 3 meses** del rango contra la de los **últimos 3 meses** para los 15 modelos más transferidos en el último trimestre, y calculá la variación porcentual.

> **Límite importante:** con 9 meses de un solo año esto muestra una **dirección**, no una proyección. Para decir "seguirá siendo la tendencia" hacen falta más años (tarea futura: cargar 2018-2025) y descontar la estacionalidad. Tampoco hay precios ni stock: es volumen de reventa, no demanda ni oferta total.

*Pista: `SUM(CASE WHEN mes <= ... THEN n END)` para cada ventana. `ADD_MONTHS(mes_min, 2)` y `ADD_MONTHS(mes_max, -2)` marcan los extremos. `NULLIF(..., 0)` evita dividir por cero.*

In [0]:
%sql
WITH n AS (
  SELECT
    automotor_marca_descripcion AS marca,
    modelo_base,
    mes,
    SUM(peso)                   AS n
  FROM transf_clean
  GROUP BY automotor_marca_descripcion, modelo_base, mes
),
lim AS (
  SELECT MIN(mes) AS prim, MAX(mes) AS ult FROM transf_clean
),
agg AS (
  SELECT
    n.marca,
    n.modelo_base,
    SUM(CASE WHEN n.mes <= ADD_MONTHS(l.prim, 2)  THEN n.n END) AS primeros_3m,
    SUM(CASE WHEN n.mes >= ADD_MONTHS(l.ult, -2)  THEN n.n END) AS ultimos_3m
  FROM n
  CROSS JOIN lim l
  GROUP BY n.marca, n.modelo_base
)
SELECT
  marca,
  modelo_base,
  ROUND(primeros_3m) AS primeros_3m,
  ROUND(ultimos_3m)  AS ultimos_3m,
  ROUND((ultimos_3m - primeros_3m) * 100.0 / NULLIF(primeros_3m, 0), 1) AS var_pct
FROM agg
ORDER BY ultimos_3m DESC
LIMIT 15;

-- Hallazgos:
-- no se como interpetar esto

---
# PARTE 7: Documentación de hallazgos

### Ejercicio 7.1: Resumen ejecutivo
Creá un resumen que documente:
1. Total de registros analizados y de transferencias estimadas
2. Principales problemas de calidad encontrados
3. Porcentaje de datos válidos vs inválidos
4. Recomendaciones para la limpieza en Silver
5. Qué se puede y qué no se puede responder con este dataset (no hay precio ni patente)

*Pista: combiná los resultados de los ejercicios anteriores en un resumen estructurado.*

## Resumen ejecutivo (completar acá)

**1. Total de registros analizados:**
-

**2. Principales problemas de calidad:** años para personas fisicas vs años en persons jurdíccas. Ignorar lo de filtrar por percentiles-
-

**3. % de datos válidos vs inválidos:** 
-

**4. Recomendaciones para Silver:**
-

**5. Qué responde y qué no responde este dataset:**
-

In [0]:
%sql
-- (Opcional) Query de apoyo para el resumen: cobertura de filas válidas
SELECT
  COUNT(*)                                                          AS filas,
  COUNT_IF(tramite_fecha IS NOT NULL
           AND fecha_inscripcion_inicial IS NOT NULL
           AND fecha_inscripcion_inicial <= tramite_fecha
           AND automotor_anio_modelo BETWEEN 1950 AND YEAR(tramite_fecha) + 1
           AND automotor_marca_descripcion IS NOT NULL
           AND titular_porcentaje_titularidad > 0
           AND titular_porcentaje_titularidad <= 100)               AS filas_validas
FROM transf_clean;

---
# ✅ Checklist de finalización
- [ ] Probé la lectura de todos los CSV antes de cargar (Parte 0)
- [ ] Creé la tabla Bronze y verifiqué que el total coincide (Parte 0)
- [ ] Completé la exploración inicial (Parte 1)
- [ ] Analicé valores nulos, vacíos y 'No aplica' (Parte 2)
- [ ] Analicé distribuciones categóricas (Parte 3)
- [ ] Calculé estadísticas descriptivas (Parte 4)
- [ ] Identifiqué problemas de calidad (Parte 5)
- [ ] Apliqué Window Functions al ranking mensual (Parte 6)
- [ ] Documenté mis hallazgos y las reglas candidatas para Silver (Parte 7)
- [ ] Subí mi notebook a GitHub
- [ ] Creé un README explicando los hallazgos principales

**Tips:** el EDA es iterativo · documentá todo · usá CTEs · las Window Functions sirven para análisis comparativos · ojo con los valores placeholder (`No aplica`, 0, vacíos) · un `NULL` en una comparación no se cuenta en `COUNT_IF`.

**Pendiente para más adelante (no es parte de este notebook):** cargar los CSV de otros años (2018-2025), automatizar la descarga del ZIP y la carga de meses nuevos con `COPY INTO`.